# CropHarvest — Exploratory Data Analysis

Repository: https://github.com/nasaharvest/cropharvest · Paper: [Tseng et al., *NeurIPS Datasets and Benchmarks* 2021](https://datasets-benchmarks-proceedings.neurips.cc/paper/2021/hash/5dd9db5e033da9c6fb5ba83c7a7ebea9-Abstract-round2.html) · Zenodo: https://zenodo.org/records/10251170 (concept DOI [10.5281/zenodo.5021761](https://doi.org/10.5281/zenodo.5021761), licence CC-BY-SA-4.0)

**Dataset in one sentence.** 113,893 globally distributed agricultural labels, harmonised from 27 source collections across 182 countries, each paired with a single-pixel, twelve-timestep multi-sensor feature array that stacks Sentinel-2, Sentinel-1, ERA5 and SRTM, designed as a benchmark for crop and non-crop mapping under label scarcity.

**Why this notebook exists.** CropHarvest is the thesis's candidate *secondary* dataset. Its role is to stress the Phase 1 pipeline under conditions the primary dataset (EuroCropsML: Estonia, Latvia, Portugal) cannot supply, namely smallholder field sizes, tropical and sub-tropical crop calendars, and geographies outside Europe. The notebook therefore asks the same class-design questions that `01_eurocropsml_eda.ipynb` asked, and then adds a section on comparability, because a secondary dataset is only useful if a single pipeline can consume both.

**What this notebook does.** All heavy work is done by two scripts, which cache their results so the notebook is fast and reproducible:

| Script | What it does | Cached artefact |
|---|---|---|
| [`_run_analysis.py`](../results/eda/cropharvest/_run_analysis.py) | catalogue, class design, figures 01 to 15, the class scheme | `catalogue_labels.parquet`, `class_inventory.parquet`, `phenology.parquet`, `findings.json`, `run_config.json`, `configs/class_scheme_cropharvest.yaml` |
| [`_offset_scan.py`](../results/eda/cropharvest/_offset_scan.py) | exhaustive label-to-pixel offset check over every one of the 87,464 arrays | `offset_scan.parquet`, `offset_scan.json` |

The code cells below load those caches and recompute them from the raw data if a cache is absent. Every number quoted in a markdown cell is printed by a code cell in this notebook, and each states whether it comes from the full dataset (113,893 labels, or all 87,464 arrays for the offset scan) or from the stratified array sample of 11,570 instances.

**Verified on-disk format** (Zenodo record 10251170, checked against the files themselves rather than the documentation):

- `labels.geojson`, 81.7 MB, EPSG:4326, 113,893 features: 77,226 `Point`, 36,666 `Polygon`, one null geometry. Columns: `index`, `dataset`, `lat`, `lon`, `is_crop`, `label`, `classification_label`, `collection_date`, `export_end_date`, `planting_date`, `harvest_date`, `is_test`, `externally_contributed_dataset`, `geometry`.
- `features/arrays/<index>_<dataset>.h5`, 87,464 files of exactly 7,872 bytes each. One HDF5 dataset `array` of shape `(12, 18)` and dtype `float64`, plus attributes `dataset`, `label`, `is_crop`, `label_lat`, `label_lon`, `instance_lat`, `instance_lon`. `(index, dataset)` joins onto `labels.geojson`.
- The 18 channels are in **fixed order**: `VV`, `VH` (Sentinel-1 GRD backscatter, dB); `B2`, `B3`, `B4`, `B5`, `B6`, `B7`, `B8`, `B8A`, `B9`, `B11`, `B12` (Sentinel-2 **L1C top-of-atmosphere** reflectance scaled by 10,000, with B1 and B10 dropped by the CropHarvest engineer); `temperature_2m` (kelvin), `total_precipitation` (metres) from ERA5; `elevation` (metres), `slope` (degrees) from SRTM, constant across all twelve timesteps; and `NDVI`, derived from B8 and B4.
- The 12 timesteps are 30-day composites. The window **ends on `export_end_date`, which is 1 February for every single label**, so it runs from 1 February of the preceding year and is anchored on a fixed global date rather than on a local growing season.
- `array` holds the value of the **single 10 m pixel nearest the label coordinate**, not an aggregate over the polygon. This is verified in section 2 and it is the single most consequential property of the dataset for this thesis.
- Missing observations are already imputed: the CropHarvest engineer replaces any remaining NaN with that band's temporal mean and discards instances where a whole band is missing. The released arrays therefore contain no NaN and carry no validity mask.

## 0. Setup

In [1]:
# %pip install pandas numpy geopandas h5py matplotlib seaborn pyarrow pyyaml
# NOTE: do NOT `pip install cropharvest` into this environment. Version 0.7.0 pins
# pandas<2.0.0 and geopandas==0.9.0, which conflicts with the pandas 2.3.3 and
# geopandas 1.1.3 this thesis uses. The dataset is read with plain h5py/geopandas.

In [2]:
from __future__ import annotations
import os, sys, json, random, subprocess
from pathlib import Path

import numpy as np, pandas as pd, geopandas as gpd, h5py
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(context="notebook", style="whitegrid")
random.seed(42); np.random.seed(42)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)


def find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "CLAUDE.md").exists() or (p / ".git").exists():
            return p
    raise RuntimeError("repository root not found")


REPO = find_repo(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("CROPHARVEST_DATA", REPO / "data" / "cropharvest")).expanduser()
EDA = REPO / "results" / "eda" / "cropharvest"
FIG = EDA / "figures"
LABELS_PATH = DATA_ROOT / "labels.geojson"
ARRAYS_DIR = DATA_ROOT / "features" / "arrays"


def cached(path: Path, script: str):
    # Load a cached artefact, computing it with its script first if it is missing.
    if not path.exists():
        print(f"{path.name} absent, running {script} ...")
        subprocess.run([sys.executable, str(EDA / script)], check=True, cwd=str(REPO))
    if path.suffix == ".json":
        return json.loads(path.read_text(encoding="utf-8"))
    return pd.read_parquet(path)


print("repository :", REPO)
print("DATA_ROOT  :", DATA_ROOT)
print("labels     :", LABELS_PATH.exists(),
      f"{LABELS_PATH.stat().st_size/1e6:.1f} MB" if LABELS_PATH.exists() else "")
print("arrays dir :", ARRAYS_DIR.is_dir())

repository : C:\Users\darey\OneDrive - University of Twente\Documents\THESIS - ExplainedGMF4Agri
DATA_ROOT  : C:\Users\darey\OneDrive - University of Twente\Documents\THESIS - ExplainedGMF4Agri\data\cropharvest
labels     : True 85.7 MB
arrays dir : True


## 1. Download

**Route taken, and why.** The official `cropharvest` PyPI package (version 0.7.0) declares `pandas<2.0.0` and `geopandas==0.9.0`. Installing it would downgrade the shared thesis environment, which runs pandas 2.3.3 and geopandas 1.1.3, so the package was **not** installed. The artefacts were fetched directly from Zenodo and are read with `h5py`, `geopandas` and `pandas` alone. Nothing in this notebook imports `cropharvest`.

**What was downloaded** (Zenodo record 10251170, the same record `cropharvest/config.py` pins as `DATASET_VERSION_ID`):

| File | Size | MD5 verified | Purpose |
|---|---:|---|---|
| `labels.geojson` | 81.7 MB | `54a5070f103bc3e635afba27c139ac8d` | all 113,893 labels and their metadata |
| `features.tar.gz` | 78.7 MB | `d757e6c32cb6d65aa517f003607f6f81` | extracts to 87,464 `.h5` arrays, 656.6 MB |

Each file took roughly 22 seconds at about 3.8 MB/s; extracting the 87,464 HDF5 files took 48 seconds. Total on disk is 829.7 MB across 87,468 files.

**Deliberately not downloaded.** `eo_data.tar.gz` (26.7 GB) holds the raw per-label GeoTIFF exports from which `features/` was derived, and is not needed because the derived arrays are the released product. `test_features.tar.gz` (786 MB) holds the arrays for the held-out benchmark regions; the corresponding labels are still present in `labels.geojson` through the `is_test` column, which is all the exploratory analysis needs.

```bash
mkdir -p data/cropharvest && cd data/cropharvest
curl -L -o labels.geojson  "https://zenodo.org/api/records/10251170/files/labels.geojson/content"
curl -L -o features.tar.gz "https://zenodo.org/api/records/10251170/files/features.tar.gz/content"
tar -xzf features.tar.gz          # creates ./features/arrays/ with 87,464 .h5 files
mkdir -p aux && curl -L -o aux/ne_10m_admin_0_countries.geojson \
  "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_0_countries.geojson"
```

The Natural Earth 1:10m admin-0 layer is an auxiliary download, needed because `labels.geojson` carries no country field. CropHarvest's own package ships the coarser 1:50m version; the finer layer is used here to reduce misassignment of coastal points.

## 2. On-disk inspection

Open the label file and one feature array before doing anything else, and confirm the schema rather than trusting the documentation.

In [3]:
labels_raw = gpd.read_file(LABELS_PATH)
print(f"{len(labels_raw):,} labels, CRS {labels_raw.crs}")
print("\ngeometry types:")
print(labels_raw.geometry.geom_type.value_counts(dropna=False).to_string())
print(f"null geometries: {labels_raw.geometry.isna().sum()}")
print("\ncolumns and dtypes:")
print(labels_raw.dtypes.to_string())

113,893 labels, CRS EPSG:4326

geometry types:
Point      77226
Polygon    36666
None           1
null geometries: 1

columns and dtypes:
harvest_date                      datetime64[ms]
planting_date                     datetime64[ms]
label                                     object
classification_label                      object
index                                      int32
is_crop                                    int32
lat                                      float64
lon                                      float64
dataset                                   object
collection_date                   datetime64[ms]
export_end_date                   datetime64[ms]
externally_contributed_dataset              bool
is_test                                     bool
geometry                                geometry


In [4]:
BANDS = ["VV", "VH",
         "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12",
         "temperature_2m", "total_precipitation", "elevation", "slope", "NDVI"]
SENSOR = ["Sentinel-1"]*2 + ["Sentinel-2 L1C"]*11 + ["ERA5"]*2 + ["SRTM"]*2 + ["derived"]
UNITS = ["dB"]*2 + ["reflectance x 1e4"]*11 + ["K", "m"] + ["m", "degrees"] + ["unitless"]

h5s = sorted(ARRAYS_DIR.glob("*.h5"))
print(f"{len(h5s):,} .h5 files, all {h5s[0].stat().st_size:,} bytes")
p = ARRAYS_DIR / "1137_togo.h5"
with h5py.File(p, "r") as h:
    print(f"\n{p.name}")
    print("  datasets:", {k: (h[k].shape, str(h[k].dtype)) for k in h.keys()})
    print("  attrs   :", {k: (round(float(v), 6) if isinstance(v, (float, np.floating)) else v)
                          for k, v in h.attrs.items()})
    arr = np.asarray(h["array"])

print(f"\narray shape {arr.shape}, dtype {arr.dtype}, NaN count {np.isnan(arr).sum()}")
print("\nper-channel summary of this one instance, and whether the channel is static:")
print(pd.DataFrame({
    "band": BANDS, "sensor": SENSOR, "unit": UNITS,
    "t0": arr[0].round(3), "min": arr.min(0).round(3), "max": arr.max(0).round(3),
    "constant over time": (arr.std(0) == 0),
}).to_string(index=False))

87,464 .h5 files, all 7,872 bytes

1137_togo.h5
  datasets: {'array': ((12, 18), 'float64')}
  attrs   : {'dataset': 'togo', 'instance_lat': 7.226812, 'instance_lon': 1.625187, 'is_crop': np.int64(0), 'label': nan, 'label_lat': 7.226807, 'label_lon': 1.625221}

array shape (12, 18), dtype float64, NaN count 0

per-channel summary of this one instance, and whether the channel is static:
               band         sensor              unit       t0      min      max  constant over time
                 VV     Sentinel-1                dB   -2.366   -4.060   -1.941               False
                 VH     Sentinel-1                dB   -4.737   -6.850   -3.725               False
                 B2 Sentinel-2 L1C reflectance x 1e4 1437.000 1388.000 2811.000               False
                 B3 Sentinel-2 L1C reflectance x 1e4 1481.000 1357.000 2599.000               False
                 B4 Sentinel-2 L1C reflectance x 1e4 1697.000 1484.000 2534.000               False
           

**Finding 1. The channel order is exactly as declared in `cropharvest/bands.py`, and the two SRTM channels are static.**

The array is `(12, 18)` `float64`. Reading one Togo instance confirms the ordering empirically: channels 0 and 1 are negative and in the range typical of Sentinel-1 backscatter in dB; channels 2 to 12 sit in the low thousands, consistent with Sentinel-2 reflectance scaled by 10,000; channel 13 is near 300, which is kelvin; channel 14 is a small fraction of a metre, which is monthly total precipitation; channels 15 and 16 are identical at every timestep, which is what `elevation` and `slope` must be; and channel 17 lies in the NDVI range. Eleven Sentinel-2 bands are present, not thirteen, because the CropHarvest engineer removes B1 and B10.

**Implication.** A pipeline consuming CropHarvest sees a wider but shallower input than one consuming EuroCropsML. It gains radar and climate covariates, and it loses two Sentinel-2 bands and, far more importantly, all sub-monthly temporal detail.

In [5]:
with h5py.File(p, "r") as h:
    a = h.attrs
    print("label coordinate    :", round(float(a["label_lat"]), 6), round(float(a["label_lon"]), 6))
    print("instance coordinate :", round(float(a["instance_lat"]), 6), round(float(a["instance_lon"]), 6))
    d_lat = abs(float(a["instance_lat"]) - float(a["label_lat"])) * 111_320
    d_lon = (abs(float(a["instance_lon"]) - float(a["label_lon"])) * 111_320
             * np.cos(np.radians(float(a["label_lat"]))))
    print(f"offset              : {d_lat:.2f} m north-south, {d_lon:.2f} m east-west")

# Exhaustive check over every array: how far is the sampled pixel from its label?
off = cached(EDA / "offset_scan.parquet", "_offset_scan.py")
OS_ = cached(EDA / "offset_scan.json", "_offset_scan.py")
print(f"\nexhaustive scan of all {OS_['arrays_scanned']:,} arrays "
      f"(tolerance {OS_['tolerance_m']} m, one 10 m pixel diagonal):")
for k in ["median_m", "p95_m", "p99_m", "max_km"]:
    print(f"  {k:10s} {OS_[k]:>14,.2f}")
print(f"  beyond tolerance : {OS_['n_bad']:,} of {OS_['arrays_scanned']:,} ({OS_['pct_bad']} %)")
print(f"  beyond 1 km      : {OS_['n_over_1km']:,} ({OS_['pct_over_1km']} %)")
print("\nevery source dataset with any affected array (full scan):")
per = pd.DataFrame(OS_["per_dataset"]).set_index("dataset")
print(per[per.n_bad > 0].to_string())
print(f"\nsource datasets entirely unaffected: {(per.n_bad == 0).sum()} of {len(per)}")
print("source datasets 100 % affected:", OS_["fully_affected_datasets"])

label coordinate    : 7.226807 1.625221
instance coordinate : 7.226812 1.625187
offset              : 0.56 m north-south, 3.76 m east-west

exhaustive scan of all 87,464 arrays (tolerance 14.14 m, one 10 m pixel diagonal):
  median_m             3.72
  p95_m                6.59
  p99_m           18,412.29
  max_km           1,456.80
  beyond tolerance : 4,062 of 87,464 (4.64 %)
  beyond 1 km      : 2,910 (3.33 %)

every source dataset with any affected array (full scan):
              n  n_bad   median_m   max_km  pct_bad
dataset                                            
uganda      225    225   82399.28   266.51    100.0
tanzania    379    379  129289.06   290.24    100.0
jecam     12856   3458       4.56  1456.75     26.9

source datasets entirely unaffected: 24 of 27
source datasets 100 % affected: ['tanzania', 'uganda']


**Finding 2. The released feature array is a single 10 m pixel at the label coordinate, and the polygon geometry is never used to aggregate it.**

Across all 87,464 arrays the median distance between the stored label coordinate and the stored instance coordinate is 3.72 m and the 95th percentile is 6.59 m, both inside one 10 m pixel, which is consistent with nothing more than snapping to the nearest pixel centre. This matches `Engineer.process_single_file` in the CropHarvest source, which selects `da.sel(x=closest_lon).sel(y=closest_lat)` from a 160 m export window and keeps that one pixel. The label's `lat` and `lon` are the polygon centroid when the geometry is a polygon.

**Implication, and it is the central one for this thesis.** "Polygon-labelled subset" in CropHarvest means *a label whose footprint is recorded as a polygon*, not *a per-parcel aggregated time series*. EuroCropsML supplies a spatial median over all pixels inside the parcel; CropHarvest supplies one pixel at the centroid. Restricting to polygons therefore buys a reliable area attribute and better confidence that the label applies to a homogeneous field, but it does **not** buy the per-parcel aggregation the proposal's pipeline assumes. Reproducing that aggregation would require re-running the Earth Engine export over each polygon, which is a substantial piece of work outside the released artefacts.

**Finding 2b, which was not expected and is not documented upstream. For 4,062 arrays (4.64 % of all 87,464) the sampled pixel is not at the label at all.** Of those, 2,910 (3.33 %) lie more than 1 km from their label and the worst is 1,457 km away. The defect is perfectly confined to three source datasets and 24 of the 27 sources are entirely clean:

| Source | Arrays | Affected | Share | Median offset |
|---|---:|---:|---:|---:|
| `tanzania` | 379 | 379 | 100 % | 129.3 km |
| `uganda` | 225 | 225 | 100 % | 82.4 km |
| `jecam` | 12,856 | 3,458 | 26.9 % | 4.6 m |

The mechanism follows from the CropHarvest exporter: labels are batched into shared Earth Engine exports, and `Engineer.find_nearest` then picks the nearest coordinate on *that GeoTIFF's* grid. A label lying outside its tif's extent is silently snapped to the tif edge rather than rejected, so the instance receives the spectral signature of a different place while keeping its original label. For `tanzania` and `uganda` this affects every single instance, which means those two source datasets, both of them smallholder crop-type campaigns of exactly the kind this thesis wants, are unusable as released.

**Implication.** This is a silent label-feature mismatch, not a missing value, and nothing in the released data flags it. It is, however, trivially detectable from the four coordinate attributes every `.h5` file carries, which is what `_offset_scan.py` does. **Every use of CropHarvest in this thesis must first drop any instance whose label-to-pixel offset exceeds one pixel diagonal, 14.14 m.** That filter is applied to every class-design number in this notebook from section 3 onwards.

## 3. Catalogue

The catalogue joins each label to a country and region by point-in-polygon against Natural Earth 1:10m admin-0, flags whether a usable feature array exists (present on disk **and** passing the offset check from Finding 2b), normalises the free-text crop label, and computes the polygon area on an equal-area projection.

In [6]:
cat = cached(EDA / "catalogue_labels.parquet", "_run_analysis.py")
F = cached(EDA / "findings.json", "_run_analysis.py")
RUNCFG = cached(EDA / "run_config.json", "_run_analysis.py")
print(f"catalogue: {len(cat):,} rows x {cat.shape[1]} columns   (full dataset)")
print(f"findings.json: {len(F)} keys, seed {RUNCFG['seed']}")
print("\narray availability (full dataset):")
print(f"  labels with an array on disk     {F['n_with_array']:,}")
print(f"  of those, passing the offset check {F['n_array_usable']:,}")
print(f"  lost to the offset defect          {F['n_lost_to_offset']:,}")
print("\ncolumns:", ", ".join(cat.columns))
cat[["index", "dataset", "country", "region", "gtype", "area_ha", "crop",
     "fao_group", "has_array", "offset_ok", "array_usable", "dominant_year"]].head(4)

catalogue: 113,893 rows x 36 columns   (full dataset)
findings.json: 109 keys, seed 42

array availability (full dataset):
  labels with an array on disk     87,464
  of those, passing the offset check 83,402
  lost to the offset defect          4,062

columns: harvest_date, planting_date, label, classification_label, index, is_crop, lat, lon, dataset, collection_date, export_end_date, externally_contributed_dataset, is_test, gtype, is_polygon, area_ha, country, iso_a3, subregion, region, has_array, array_path, pixel_offset_m, offset_ok, array_usable, fao_group_raw, crop, is_crop_type, fao_group, agg_group, window_end, window_start, window_end_year, dominant_year, alphaearth_ok, tessera_ok


,index,dataset,country,region,gtype,area_ha,crop,fao_group,has_array,offset_ok,array_usable,dominant_year
0,0,ethiopia,Ethiopia,Sub-Saharan Africa,Polygon,0.583604,None,None,True,True,True,2020
1,1,ethiopia,Ethiopia,Sub-Saharan Africa,Polygon,0.153480,None,None,True,True,True,2020
2,2,ethiopia,Ethiopia,Sub-Saharan Africa,Polygon,0.130357,None,None,True,True,True,2020
3,3,ethiopia,Ethiopia,Sub-Saharan Africa,Polygon,0.066848,None,None,True,True,True,2020


## 4. Part 1 — general understanding

Every figure in this part was produced by `_run_analysis.py` on the **full** 113,893-label file. Each subsection states the finding and then what it means for the thesis.

### 4.1 Geographic spread

In [7]:
reg = cat.region.value_counts()
display(pd.DataFrame({"labels": reg, "share %": (reg / len(cat) * 100).round(1)}))
print(f"{cat.country.nunique()} distinct countries (full dataset)")
print("\ntop 12 countries:")
print(cat.country.value_counts().head(12).to_string())

,labels,share %
region,,
Sub-Saharan Africa,35942,31.6
South America,20887,18.3
Europe,14375,12.6
North America,13593,11.9
East & Southeast Asia,12835,11.3
Central Asia,6501,5.7
Central America & Caribbean,3517,3.1
South Asia,2708,2.4
MENA,2365,2.1


182 distinct countries (full dataset)

top 12 countries:


country
Brazil                        18742
Canada                         9884
People's Republic of China     9575
France                         6518
Madagascar                     5373
Uzbekistan                     5314
United States of America       3683
Tanzania                       3446
Burkina Faso                   3184
Kenya                          2945
Reunion (FR)                   2776
Senegal                        2769


![Labels per region](../results/eda/cropharvest/figures/01_labels_per_region.png)

![Label map](../results/eda/cropharvest/figures/08_label_map.png)

**Finding 3. The geographic spread is genuine and it is the dataset's strongest asset.** Across the full dataset, Sub-Saharan Africa holds 35,942 labels (31.6 %), South America 20,887 (18.3 %), Europe 14,375 (12.6 %), North America 13,593 (11.9 %), East and Southeast Asia 12,835 (11.3 %) and Central Asia 6,501 (5.7 %), spread over 182 countries. The six largest national contributors are Brazil (18,742), Canada (9,884), China (9,575), France (6,518), Madagascar (5,373) and Uzbekistan (5,314). France is counted here as metropolitan France alone, because Natural Earth merges the French overseas departments into it and this notebook splits Reunion (2,776) and Martinique (2,421) back out by source dataset, following the UN M49 placement of Reunion in Eastern Africa and Martinique in the Caribbean.

**Implication.** On geographic coverage alone, CropHarvest does exactly what the proposal asks of a secondary dataset: it places the pipeline in smallholder and tropical systems on four continents that EuroCropsML cannot reach. Whether that coverage survives the restrictions the thesis needs is the question the rest of this notebook answers.

### 4.2 Source datasets and provenance

In [8]:
src = pd.DataFrame(F["source_datasets"]).set_index("dataset")
display(src[["n", "polygons", "with_croptype", "pct_croptype", "with_array", "pct_array"]])
print(f"{F['n_source_datasets']} source datasets (full dataset)")
print(f"{F['n_labels_binary_only']:,} labels ({F['n_labels_binary_only']/len(cat)*100:.1f} %) "
      "come from sources that carry no crop type at all")

,n,polygons,with_croptype,pct_croptype,with_array,pct_array
dataset,,,,,,
geowiki-landcover-2017,35866,0,0,0.0,24761,69.0
croplands,14976,0,577,3.9,4519,30.2
jecam,13083,13083,10909,83.4,12856,98.3
canada,9088,0,6976,76.8,7160,78.8
ile-de-france,6184,6184,4492,72.6,6184,100.0
china-crop,5624,0,0,0.0,4397,78.2
central-asia,5302,5302,5074,95.7,4894,92.3
rwanda-ceo,3600,0,0,0.0,3591,99.8
reunion-france,2776,2776,1445,52.1,2238,80.6


27 source datasets (full dataset)
53,206 labels (46.7 %) come from sources that carry no crop type at all


![Source datasets](../results/eda/cropharvest/figures/02_source_datasets.png)

**Finding 4. The dataset is a union of 27 heterogeneous collections, and eleven of them carry no crop type whatsoever.** Those eleven contribute 53,206 labels, 46.7 % of the total (full dataset). The single largest source, `geowiki-landcover-2017` with 35,866 labels, is crowdsourced visual interpretation of a global grid and is binary only. The second largest, `croplands` (GFSAD) with 14,976 labels, carries a crop type for just 3.9 % of its rows and, separately, is missing a feature array for 69.8 % of them.

The remaining sources differ sharply in protocol. `canada` contributes exactly 100 labels for each of 130 classes, which is a deliberate stratified cap rather than a natural distribution. The three French RPG extracts (`ile-de-france`, `reunion-france`, `martinique-france`) are administrative subsidy declarations and their labels are untranslated French RPG codes. `jecam`, `central-asia`, `germany` and `lem-brazil` are field campaigns or national parcel registers with proper crop typing.

**Implication.** Any statement about CropHarvest label quality has to be made per source, not for the dataset as a whole. Three groups of sources should be excluded for distinct reasons: `geowiki-landcover-2017` and `croplands` for geolocation precision, since the first places labels on a regular grid by remote visual interpretation and the second aggregates contributions of mixed provenance; and `tanzania` and `uganda` for the feature-coordinate mismatch of Finding 2b, which affects every one of their instances. The first two are point-only and are removed by the polygon restriction in any case; the second two are polygon sources and would otherwise have been kept.

### 4.3 Points, polygons, and what the polygon restriction actually costs

In [9]:
print("full dataset")
print(f"  points            {F['n_points']:,}")
print(f"  polygons          {F['n_polygons']:,}  ({F['polygon_share_pct']} %)")
print(f"  null geometry     {F['n_missing_geom']}")
print(f"  polygons with a feature array on disk         {F['polygons_with_array']:,}")
print(f"  polygons with a usable crop type              {F['polygons_with_croptype']:,}")
print(f"  polygons with BOTH an array and a crop type   {F['polygons_with_array_and_croptype']:,}")
print(f"  of those, lost to the offset defect           {F['polygons_lost_to_offset']:,}")
print(f"  FULLY USABLE polygon crop-type labels         {F['polygons_usable']:,}")
print("\npolygons per region (full dataset):")
print(pd.Series(F["polygons_per_region"]).to_string())
print("\npolygon area, hectares (full dataset):")
print(pd.Series(F["polygon_area_ha"]).to_string())
print(f"\n{F['polygons_smallholder_under_2ha_pct']} % of polygons are below 2 ha; "
      f"{F['polygons_below_1_s2_pixel']} are smaller than a single 10 m Sentinel-2 pixel")

full dataset
  points            77,226
  polygons          36,666  (32.2 %)
  null geometry     1
  polygons with a feature array on disk         35,169
  polygons with a usable crop type              25,441
  polygons with BOTH an array and a crop type   24,363
  of those, lost to the offset defect           3,892
  FULLY USABLE polygon crop-type labels         20,471

polygons per region (full dataset):
Sub-Saharan Africa             18025
Europe                          8734
Central Asia                    5302
Central America & Caribbean     2421
South America                   2139
MENA                              45

polygon area, hectares (full dataset):
count    36666.0000
mean         6.0788
std         30.7177
min          0.0000
1%           0.0142
5%           0.0342
25%          0.1706
50%          0.7207
75%          4.4646
95%         21.2795
99%         83.5929
max       2622.2814

64.5 % of polygons are below 2 ha; 220 are smaller than a single 10 m Sentinel-2 pixel


![Geometry types](../results/eda/cropharvest/figures/03_geometry_types.png)

![Polygon area](../results/eda/cropharvest/figures/04_polygon_area.png)

**Finding 5. A third of the dataset is polygon-labelled, and the polygons are genuinely smallholder in size.** Of 113,893 labels, 36,666 (32.2 %) are polygons and 77,226 are points (full dataset). The polygons are concentrated exactly where the thesis wants them: 18,025 in Sub-Saharan Africa, 8,734 in Europe, 5,302 in Central Asia, 2,421 in the Caribbean and 2,139 in South America. Fifteen of the 27 source datasets supply polygons; the flag is simply the GeoJSON geometry type, and there is no separate metadata field for it.

Median polygon area is 0.72 ha and 64.5 % of polygons are below 2 ha, which is the conventional smallholder threshold. Field size varies over five orders of magnitude by source: `lem-brazil` has a median of 54 ha (Bahian industrial agriculture), `central-asia` 6.4 ha and `germany` 3.6 ha, against `kenya-non-crop` at 0.055 ha, `togo` at 0.13 ha and `kenya` at 0.13 ha. Two hundred and twenty polygons are smaller than one 10 m Sentinel-2 pixel, which matters because the feature array is exactly one such pixel.

**Finding 6. The polygon restriction is not severe. It is, if anything, beneficial.** After restricting to polygons, requiring a feature array, requiring a usable crop type and applying the offset filter, **20,471 labels remain** (full dataset). That is 18.0 % of the raw file, but it discards almost nothing the thesis could have used: the point-only sources are overwhelmingly the binary, crowdsourced ones. Before the offset filter the figure is 24,363, so Finding 2b costs 3,892 labels, 16.0 % of the otherwise usable polygon pool. Polygons retain 25,441 of the 35,349 crop-typed labels, or 72.0 %.

**Implication.** The polygon restriction in the proposal is the right call and it survives contact with the data. It removes the two sources flagged for geolocation precision at no cost to the crop-typed pool, it supplies an area attribute with which to stratify results by field size, and it makes a smallholder-versus-commercial contrast possible within one dataset. It does not, however, convert the single-pixel feature array into a parcel aggregate, as Finding 2 established.

### 4.4 Class inventory

In [10]:
inv = cached(EDA / "class_inventory.parquet", "_run_analysis.py").set_index("crop")
print(f"full dataset: {F['n_raw_labels']} distinct raw label strings collapse to "
      f"{F['n_distinct_crops']} usable crop types over {F['n_with_croptype']:,} labels")
print(f"labels with no usable crop type: {len(cat) - F['n_with_croptype']:,}")
print("\nexcluded by each rule (full dataset):")
print(pd.Series(F["excluded_label_counts"]).to_string())
print("\ntop 20 crop types (n, contributing source datasets, countries, regions, polygons):")
display(inv.head(20)[["n", "share_pct", "n_datasets", "n_countries", "n_regions", "n_polygons"]])
print("\ncrop-typed labels per region (full dataset):")
print(pd.Series(F["croptype_labels_per_region"]).to_string())
print("\ncrop-typed labels per country, top 15 (full dataset):")
print(pd.Series(F["croptype_labels_per_country_top20"]).head(15).to_string())
print("\nFAO indicative crop classification groups (full dataset):")
print(pd.Series(F["fao_group_counts"]).to_string())
print("\nFAO assignments this analysis flags as defective (full dataset):")
print(pd.DataFrame(F["fao_group_defects"]).to_string(index=False))

full dataset: 391 distinct raw label strings collapse to 272 usable crop types over 35,349 labels
labels with no usable crop type: 78,544

excluded by each rule (full dataset):
non_crop_fao_group            5902
french_autre_catchall         1671
curated_non_crop_type_list    6979

top 20 crop types (n, contributing source datasets, countries, regions, polygons):


,n,share_pct,n_datasets,n_countries,n_regions,n_polygons
crop,,,,,,
cotton,3871,10.95,5,7,4,3860
maize,3788,10.72,13,19,7,2984
rice,2917,8.25,6,11,5,1432
wheat,2248,6.36,4,6,4,1860
millet,1295,3.66,7,8,5,922
sorghum,960,2.72,7,8,4,698
groundnut,956,2.70,2,5,1,956
soybean,937,2.65,5,8,5,837
sugarcane,842,2.38,5,5,4,785



crop-typed labels per region (full dataset):
Sub-Saharan Africa             15003
North America                   6979
Europe                          5537
Central Asia                    5074
Central America & Caribbean     1295
South America                    887
East & Southeast Asia            399
MENA                             141
South Asia                        34

crop-typed labels per country, top 15 (full dataset):
Canada             6960
Uzbekistan         4980
France             4492
Madagascar         4350
Burkina Faso       2819
Senegal            2504
Mali               1654
Reunion (FR)       1445
Martinique (FR)    1295
Tanzania           1185
Germany            1045
Brazil              887
South Africa        461
Thailand            369
Kenya               304

FAO indicative crop classification groups (full dataset):
cereals              14087
other                 9139
non_crop              5902
fruits_nuts           4807
vegetables_melons     4284
oilseeds    

![Crop type inventory](../results/eda/cropharvest/figures/05_croptype_inventory.png)

![FAO group by region](../results/eda/cropharvest/figures/06_fao_group_by_region.png)

**Finding 7. Only 35,349 of 113,893 labels (31.0 %) name a crop, and the inventory has a long tail of 272 types.** The remainder are removed by three auditable rules: 5,902 labels whose own FAO group is `non_crop`, 6,979 matched by a curated list of land-cover rather than crop terms (meadow, pasture, fallow, urban, water, forestry), and 1,671 French RPG catch-all categories beginning with *autre* (French for "other").

The top of the inventory is dominated by six crops: cotton (3,871), maize (3,788), rice (2,917), wheat (2,248), millet (1,295) and sorghum (960). Maize is the most widely distributed, appearing in 13 source datasets, 19 countries and 7 regions. Cotton is the opposite: 3,871 labels but almost all from Uzbekistan through the `central-asia` source. By region, crop-typed labels concentrate in Sub-Saharan Africa (15,003), North America (6,979), Europe (5,537) and Central Asia (5,074), and fall away sharply after that.

Only 46,288 labels carry the harmonised `classification_label`, and the largest group inside it after `cereals` (14,087) is `other` (9,139), which is a residual bin rather than an agronomic class.

**Finding 8. The FAO harmonisation contains demonstrable defects affecting 4,784 labels (full dataset).** Cotton, a fibre crop, has no home in the ICC11 core list and is silently pooled into `other` (3,871 labels). `pine` is assigned to `fruits_nuts` and `eucalyptus` to `other`, although both are forestry rather than agriculture (713 labels between them). `potato` is assigned to `root_tuber` by one source and `vegetables_melons` by another; `sugarcane` maps to `sugar` while `sugarbeet` maps to `vegetables_melons`. Separately, the French RPG labels distinguish *fermage* (tenanted) from *propriété ou faire valoir direct* (owner-occupied) for the same crop, so `canne à sucre` and `banane créole` each appear as three distinct label strings that differ only in land tenure.

**Implication.** The FAO `classification_label` cannot be used as-is as a target. The thesis needs its own normalisation layer, which is what the `LABEL_NORM` and `CROP_TO_AGGREGATE` maps in `_run_analysis.py` provide and what `configs/class_scheme_cropharvest.yaml` records. This is not unlike the HCAT situation in EuroCropsML, but it is worse, because HCAT is a single consistently applied hierarchy whereas this is a post-hoc reconciliation of twenty-seven schemes.

### 4.5 Label years, and how the observation window is anchored

In [11]:
print("export_end_date, month and day (full dataset):")
print(pd.Series(F["export_end_month_day"]).to_string())
print(f"\nobservation window spans {F['window_span'][0]} to {F['window_span'][1]}")
print("\ncalendar year holding 11 of the 12 monthly timesteps (full dataset):")
print(pd.Series(F["dominant_year_counts"]).to_string())
print(f"\nsame, restricted to the usable polygon crop-type subset "
      f"(n={F['polygon_croptype_n']:,}):")
print(pd.Series(F["polygon_croptype_dominant_years"]).to_string())
print("\nfield collection year (full dataset):")
print(pd.Series(F["collection_year_counts"]).to_string())

export_end_date, month and day (full dataset):
02-01    113893

observation window spans 2015-02-06 to 2022-02-01

calendar year holding 11 of the 12 monthly timesteps (full dataset):
2015      498
2016    55127
2017    12945
2018    14271
2019    20225
2020     6264
2021     4563

same, restricted to the usable polygon crop-type subset (n=20,471):
2016    2427
2017    2797
2018    5790
2019    9225
2020     225
2021       7

field collection year (full dataset):
2016    39196
2017    11026
2018     9558
2019    17738
2020     7527
2021    23830
2022     5018


![Label years](../results/eda/cropharvest/figures/07_label_years_coverage.png)

**Finding 9. The twelve-month window is anchored on a fixed global date, 1 February, for all 113,893 labels, and is therefore not aligned to any local growing season.** `export_end_date` is `02-01` for every row without exception. The window runs 360 days back from there, so timestep 0 is February and timestep 11 is January. CropHarvest's own configuration explains the choice: the comment in `cropharvest/config.py` states that the export runs "until the 1st February, at which point planting for the long rains can start", which aligns the window to the **East African** long-rains calendar and then applies that same anchor to Canada, France, Uzbekistan and Brazil alike.

Label years span 2015 to 2022, concentrated on 2016 (55,127 labels, almost all `geowiki-landcover-2017`), 2019 (20,225), 2018 (14,271) and 2017 (12,945). Field collection dates run from 2016 to 2022 and frequently differ from the observation year.

**Implication.** This cuts in two directions. A fixed February anchor is *better* than a calendar-year anchor for Southern Hemisphere and equatorial crops, whose seasons straddle 1 January and would be split in half by a January-to-December window. It is *worse* for Northern Hemisphere winter crops, whose autumn sowing falls in the previous window. Most importantly for this thesis, it is **misaligned with every annual embedding product**, all of which are calendar-year: AlphaEarth's 2021 image covers 1 January 2021 to 1 January 2022, whereas a CropHarvest label with `export_end_date` 2021-02-01 covers 1 February 2020 to 27 January 2021. Section 8 quantifies the damage.

### 4.6 Spatial duplication

In [12]:
dup = pd.DataFrame(F["near_duplicates"]).T
dup.index.name = "radius (m)"
display(dup)
print(f"exact coordinate duplicates (identical lat and lon): {F['n_exact_coordinate_duplicates']:,} "
      f"({F['n_exact_coordinate_duplicates']/len(cat)*100:.1f} % of the full dataset)")

,n_pairs,n_cross_dataset_pairs,n_labels_involved,pct_labels_involved
radius (m),,,,
10,9884.0,22.0,10447.0,9.17
100,40373.0,136.0,24801.0,21.78
1000,570547.0,2625.0,61254.0,53.78


exact coordinate duplicates (identical lat and lon): 5,102 (4.5 % of the full dataset)


**Finding 10. Near-duplication is substantial within sources and negligible across them.** Within 10 m, 9,884 label pairs exist, involving 10,447 labels (9.2 % of the full dataset), but only 22 of those pairs cross a source-dataset boundary. Within 100 m, 40,373 pairs involve 24,801 labels (21.8 %), of which 136 pairs cross sources. There are 5,102 labels sharing an exactly identical coordinate with another label.

**Implication.** The risk here is not double-counting the same field from two independent surveys; that essentially does not happen, so no cross-source deduplication step is needed. The risk is **spatial autocorrelation within a source**: clusters of labels a few tens of metres apart in the same field, which will land in both the training and the test split under random partitioning and will inflate every score. This is the same hazard the proposal already plans for with spatial block cross-validation (Roberts et al., 2017), and the numbers above say the blocks must be at least 100 m and preferably 1 km, at which radius 53.8 % of labels have a neighbour.

## 5. Part 2 — class design

This part answers, for CropHarvest, the three questions `01_eurocropsml_eda.ipynb` answered for EuroCropsML: what classes exist, what must be cut, and what aggregation is needed. Every count from here on requires a label to have a crop type **and** a feature array **and** to pass the offset check of Finding 2b.

### 5.1 What needs to be cut off

The K-shot protocol in the proposal requires, per class and per region, enough labels to draw at least five independent training subsets at the largest budget and still hold out a disjoint test set. Taking the largest budget as 100 samples per class and requiring at least 50 held-out samples gives a working floor of **150 labels per class per region**. The cell below reports how many classes survive at each candidate threshold.

In [13]:
surv = pd.DataFrame(F["class_survival"])
display(surv.pivot(index="threshold", columns="subset",
                   values=["n_region_class_pairs", "n_distinct_crops", "n_labels"]))
print(f"floor adopted: {F['class_floor']} labels per (region, crop)")
print("\ncrops meeting the floor, per region (full dataset, usable arrays only):")
for r, v in F["crops_meeting_floor_per_region"].items():
    print(f"  {r:30s} {len(v):2d}  {', '.join(v) if v else '(none)'}")

n_region_class_pairs               n_distinct_crops                     n_labels              
subset          all geometries polygons only   all geometries polygons only all geometries polygons only
threshold                                                                                               
20                         186           121              144           100          27635         19699
50                         130            77              100            62          25784         18284
100                         84            54               67            44          22300         16623
150                         24            21               15            14          16035         13059
200                         23            20               15            14          15865         12889
300                         15            14               12            11          14123         11534
500                          5             4                5             4          10089          7649
1000                         4             4                4             4           9425          7649

floor adopted: 150 labels per (region, crop)

crops meeting the floor, per region (full dataset, usable arrays only):
  Sub-Saharan Africa             12  banana, carrot, cassava, cotton, groundnut, maize, millet, potato, rice, sorghum, soybean, sugarcane
  Central Asia                    2  cotton, wheat
  Europe                          4  maize, potato, rye, wheat
  North America                   3  barley, maize, wheat
  Central America & Caribbean     2  banana, sugarcane
  South America                   1  sugarcane
  East & Southeast Asia           0  (none)
  MENA                            0  (none)
  South Asia                      0  (none)


![Class survival](../results/eda/cropharvest/figures/09_class_survival.png)

**Finding 11. At a 150-label floor, 21 (region, crop) pairs survive across the whole world in the polygon subset, covering 14 distinct crops and 13,059 labels.** Lowering the floor to 100 raises this to 54 pairs and 44 crops, and to 50 raises it to 77 pairs and 62 crops, but those thresholds cannot support five independent draws at a 100-shot budget plus a held-out set.

Recommended exclusion rules, each with the number behind it (all from the full dataset):

| Rule | Labels removed | Rationale |
|---|---:|---|
| Source carries no crop type | 53,206 | eleven binary-only sources, 46.7 % of the file |
| No feature array on disk | 26,429 | 23.2 % of labels; concentrated in `croplands` (69.8 % missing) and `geowiki-landcover-2017` (31.0 %) |
| Label-to-pixel offset above 14.14 m | 4,062 arrays, 3,892 of them in the polygon crop-typed pool | silent label-feature mismatch, Finding 2b; removes `tanzania` and `uganda` entirely and 26.9 % of `jecam` |
| Label is not a crop type | 14,552 | 5,902 with FAO group `non_crop`, 6,979 land-cover terms, 1,671 French *autre* catch-alls |
| Not a polygon | 77,226 | the proposal's restriction; also removes both sources flagged for geolocation precision |
| Class below 150 per region | remainder | leaves 13,059 labels in 21 (region, crop) pairs |
| Observation window before 2017 | 2,427 of the usable polygon crop-typed subset | outside AlphaEarth coverage, see section 8 |

On missing or degraded series there is an important negative result. The released arrays contain **zero** NaN values and the per-instance count of time-constant channels is 1.997 on average across the 11,570-instance sample, which is the two SRTM channels and nothing else. Cloud-affected timesteps cannot be detected from the released data, because the CropHarvest engineer has already replaced them with the band's own temporal mean and has dropped the instances where a whole band was missing. There is no validity mask. A cloud-quality filter is therefore **not available** on this dataset, and the 26,429 missing arrays are the only visible trace of the export having failed.

On label quality, four sources should be excluded. `geowiki-landcover-2017` (35,866 labels, crowdsourced grid interpretation) and `croplands` (14,976 labels, mixed provenance, 3.9 % crop-typed) fail on geolocation precision, and both are point-only. `tanzania` (379) and `uganda` (225) fail on the feature-coordinate mismatch, and both are polygon sources that would otherwise have been valuable. The label-year distribution is itself a quality issue: field collection dates and observation years frequently differ, so a label collected in 2021 may describe a field whose imagery window ran in 2019.

### 5.2 Aggregation, and the cross-region intersection

In [14]:
print("aggregated groups meeting the floor, per region (full dataset):")
for r, v in F["agg_groups_meeting_floor_per_region"].items():
    print(f"  {r:30s} {len(v):2d}  {', '.join(v) if v else '(none)'}")
print("\ncrops meeting the floor in two or more regions:")
print("  ", ", ".join(F["cross_region_shared_crops"]))
print("aggregated groups meeting the floor in two or more regions:")
print("  ", ", ".join(F["cross_region_shared_agg_groups"]))
print(f"\nadmissible transfer pairs (two or more shared classes): "
      f"{F['n_admissible_transfer_pairs']}")
print("\nall pairwise region intersections at the 150 floor:")
for d in F["region_pair_intersections"]:
    if d["n_shared"]:
        print(f"  {d['region_a']:30s} x {d['region_b']:30s} {d['n_shared']}  {d['shared']}")

aggregated groups meeting the floor, per region (full dataset):
  Central America & Caribbean     2  perennial_fruit, sugar
  Central Asia                    2  cereals, fibre
  East & Southeast Asia           0  (none)
  Europe                          4  cereals, legumes, oilseeds, roots_tubers
  MENA                            0  (none)
  North America                   3  cereals, oilseeds, vegetables
  South America                   1  sugar
  South Asia                      0  (none)
  Sub-Saharan Africa              8  cereals, fibre, legumes, oilseeds, perennial_fruit, roots_tubers, sugar, vegetables

crops meeting the floor in two or more regions:
   banana, cotton, maize, potato, sugarcane, wheat
aggregated groups meeting the floor in two or more regions:
   cereals, fibre, legumes, oilseeds, perennial_fruit, roots_tubers, sugar, vegetables

admissible transfer pairs (two or more shared classes): 3

all pairwise region intersections at the 150 floor:
  Sub-Saharan Africa    

![Region class intersection](../results/eda/cropharvest/figures/10_region_class_intersection.png)

![Region pair intersection](../results/eda/cropharvest/figures/15_region_pair_intersection.png)

**Finding 12, and the headline of this notebook. The cross-region transfer experiment barely has a label space to run in.** Sub-Saharan Africa alone reaches the 150-label floor on 12 crops: banana, carrot, cassava, cotton, groundnut, maize, millet, potato, rice, sorghum, soybean and sugarcane. Every other region reaches it on at most four. But the **intersection** of any two regions never exceeds **two classes** (full dataset):

| Region pair | Shared classes |
|---|---|
| Sub-Saharan Africa and Europe | maize, potato |
| Sub-Saharan Africa and Central America and Caribbean | banana, sugarcane |
| Europe and North America | maize, wheat |
| Sub-Saharan Africa and Central Asia | cotton |
| Sub-Saharan Africa and North America | maize |
| Sub-Saharan Africa and South America | sugarcane |
| Central Asia and Europe; Central Asia and North America | wheat |
| Central America and Caribbean and South America | sugarcane |

Only three pairs share two classes, and none shares three. The comparison with EuroCropsML is stark: there, eleven of the top twenty classes are present in all three countries, so a transfer experiment has a meaningful shared label space. Here, the most generous cross-region experiment CropHarvest supports is a two-class problem.

Aggregating to agronomic groups (cereals, legumes, oilseeds, roots and tubers, sugar, fibre, perennial fruit, vegetables, beverage and spice, fodder) improves matters but not decisively. Sub-Saharan Africa reaches the floor on eight groups, Europe on four, North America on three, and Central Asia, the Caribbean and South America on two, two and one respectively. Eight groups clear the floor in at least two regions, so aggregation is the only route to a cross-region experiment with more than two classes, and even then Sub-Saharan Africa is the only region that could act as a rich source domain.

**Implication.** The cross-region analogue of EuroCropsML's transnational protocol is viable only at the aggregated-group level, and only with Sub-Saharan Africa on one side. A crop-level transfer experiment would be a two-class problem, which is not an informative test of whether a foundation model has learned location-invariant agricultural representations.

### 5.3 The proposed scheme

In [15]:
import yaml
scheme = yaml.safe_load((REPO / "configs" / "class_scheme_cropharvest.yaml").read_text(encoding="utf-8"))
print("configs/class_scheme_cropharvest.yaml")
print(f"  generated {scheme['_meta']['generated_at']}, seed {scheme['_meta']['seed']}, "
      f"floor {scheme['_meta']['min_labels_per_class_per_region']}")
rows = []
for r, v in scheme["regions"].items():
    rows.append(dict(region=r, task=v["task"], labels=v["n_labels_total"],
                     usable_polygons=v["n_polygon_with_array"],
                     n_crop=len(v["crop_classes"]), n_agg=len(v["aggregated_classes"]),
                     crops=", ".join(v["crop_classes"][:5])
                             + ("..." if len(v["crop_classes"]) > 5 else "")))
display(pd.DataFrame(rows).sort_values("labels", ascending=False).set_index("region"))
print("cross-region shared crop classes      :",
      scheme["cross_region_transfer"]["shared_crop_classes"])
print("cross-region shared aggregated classes:",
      scheme["cross_region_transfer"]["shared_aggregated_classes"])
print("\nexclusion rules recorded in the scheme:")
for k, v in scheme["exclusions"].items():
    print(f"  {k}: {v if not isinstance(v, list) or len(v) < 6 else str(v[:5]) + f' ... ({len(v)} total)'}")

configs/class_scheme_cropharvest.yaml
  generated 2026-09-11T09:31:09Z, seed 42, floor 150


,task,labels,usable_polygons,n_crop,n_agg,crops
region,,,,,,
Sub-Saharan Africa,multiclass_croptype,35942,13166,12,8,"rice, maize, millet, carrot, potato..."
South America,binary_cropland,20887,2139,1,1,sugarcane
Europe,small_multiclass_croptype,14375,8734,4,4,"maize, wheat, rye, potato"
North America,small_multiclass_croptype,13593,0,3,3,"barley, maize, wheat"
East & Southeast Asia,binary_cropland,12835,0,0,0,
Central Asia,small_multiclass_croptype,6501,4894,2,2,"cotton, wheat"
Central America & Caribbean,small_multiclass_croptype,3517,2129,2,2,"banana, sugarcane"
South Asia,binary_cropland,2708,0,0,0,
MENA,binary_cropland,2365,45,0,0,


cross-region shared crop classes      : ['banana', 'cotton', 'maize', 'potato', 'sugarcane', 'wheat']
cross-region shared aggregated classes: ['cereals', 'fibre', 'legumes', 'oilseeds', 'perennial_fruit', 'roots_tubers', 'sugar', 'vegetables']

exclusion rules recorded in the scheme:
  binary_only_source_datasets: ['geowiki-landcover-2017', 'china-crop', 'rwanda-ceo', 'kenya-non-crop', 'tanzania-ceo'] ... (11 total)
  drop_labels_without_feature_array: True
  drop_label_to_pixel_offset_above_m: 14.14
  fully_mismatched_source_datasets: ['tanzania', 'uganda']
  drop_non_crop_type_labels: ['abandoned (overgrown)', 'abandoned (shrubs)', 'annual crop', 'barren', 'birdsfoot trefoil'] ... (35 total)
  drop_fao_groups: ['non_crop', 'other']
  drop_label_years_before: 2017
  merge_french_rpg_tenure_suffixes: True
  flagged_low_geolocation_precision_sources: ['geowiki-landcover-2017', 'croplands']


**Proposed task definition per region**, written to [`configs/class_scheme_cropharvest.yaml`](../configs/class_scheme_cropharvest.yaml):

- **Sub-Saharan Africa**: genuine multi-class crop typing on 12 classes. This is the only region in CropHarvest that supports it, and it is also the region with the smallest fields, so it is the correct primary target for the secondary-dataset experiment.
- **Europe, North America, Central Asia, Central America and Caribbean**: small multi-class crop typing on two to four classes each. Useful as transfer targets, not as standalone benchmarks.
- **South America, East and Southeast Asia, South Asia, MENA, Oceania**: binary cropland mapping only. They have labels, some of them many, but not crop types at sufficient density.
- **Cross-region transfer**: admissible only on the aggregated-group scheme, with the shared set `cereals, fibre, legumes, oilseeds, perennial_fruit, roots_tubers, sugar, vegetables`, and only for pairs where both regions reach the floor on every class in the intersection.

## 6. Part 3 — the feature arrays

All numbers in this part come from a **stratified random sample of 11,570 feature arrays** drawn with seed 42, capped at 230 per (region, FAO group) stratum, not from all 87,464 arrays. Reading the full set takes about 37 minutes on this machine because of OneDrive synchronisation latency, so the sample is cached to `phenology.parquet`. The one exception is the label-to-pixel offset of Finding 2b, which was scanned exhaustively because it reads only the HDF5 attributes.

### 6.1 Example time series

In [16]:
ph = cached(EDA / "phenology.parquet", "_run_analysis.py")
print(f"array sample: {len(ph):,} instances of {F['arrays_total_on_disk']:,} on disk "
      f"({len(ph)/F['arrays_total_on_disk']*100:.1f} %), seed {RUNCFG['seed']}")
print(f"NaN values across the whole sample: {F['array_nan_count']}")
print(f"mean number of time-constant channels per instance: {F['array_constant_band_mean']}")
print("\ninstances per region in the sample:")
print(ph.region.value_counts().to_string())

array sample: 11,570 instances of 87,464 on disk (13.2 %), seed 42
NaN values across the whole sample: 0
mean number of time-constant channels per instance: 1.997

instances per region in the sample:
region
Sub-Saharan Africa             2476
Europe                         2300
North America                  1840
South America                  1482
Central America & Caribbean    1302
Central Asia                    947
MENA                            371
East & Southeast Asia           365
South Asia                      257
Oceania                         230


![Maize example](../results/eda/cropharvest/figures/11_bands_example_ts_maize.png)

![Rice example](../results/eda/cropharvest/figures/11_bands_example_ts_rice.png)

![Wheat example](../results/eda/cropharvest/figures/11_bands_example_ts_wheat.png)

**Finding 13. Twelve monthly composites resolve a temperate crop cycle adequately and a tropical one poorly.** The maize panels show a recognisable single-peak signature in the temperate regions: reflectance in the near infrared rises through the middle of the window and the NDVI trace has one clear maximum. The Sub-Saharan and South American panels are visibly flatter, and the Sentinel-1 VV trace (plotted on the right-hand axis) carries at least as much structure as NDVI in those panels, which is one concrete argument for CropHarvest's inclusion of radar and one reason not to discard the radar channels lightly when building a shared pipeline.

### 6.2 Phenology by region

In [17]:
print("median seasonal NDVI amplitude (max - min) per region, array sample of "
      f"{len(ph):,}:")
print(pd.Series(F["ndvi_amp_by_region"]).sort_values(ascending=False).to_string())
print("\nmedian ERA5 annual 2 m temperature range (K), same sample:")
print(pd.Series(F["t2m_amp_by_region"]).sort_values(ascending=False).to_string())
print("\nmean-curve NDVI statistics per aggregated group, two contrasting regions:")
for r in ["Europe", "Sub-Saharan Africa"]:
    s = pd.DataFrame(F["phenology_summary"][r]).T
    print(f"\n  {r}:")
    print(s[["n", "peak_step", "peak_ndvi", "amplitude"]].to_string())

median seasonal NDVI amplitude (max - min) per region, array sample of 11,570:
North America                  0.572
Europe                         0.489
East & Southeast Asia          0.474
Oceania                        0.461
Central Asia                   0.445
South Asia                     0.397
South America                  0.375
MENA                           0.328
Central America & Caribbean    0.303
Sub-Saharan Africa             0.279

median ERA5 annual 2 m temperature range (K), same sample:
Central Asia                   28.46
North America                  27.20
East & Southeast Asia          22.02
MENA                           19.67
Europe                         15.86
South Asia                     14.63
Oceania                        13.65
South America                   7.94
Sub-Saharan Africa              6.35
Central America & Caribbean     2.50

mean-curve NDVI statistics per aggregated group, two contrasting regions:

  Europe:
                     n  peak_step  

![NDVI phenology by region](../results/eda/cropharvest/figures/12_ndvi_phenology_by_region.png)

![Array diagnostics](../results/eda/cropharvest/figures/13_array_diagnostics.png)

**Finding 14. Crop groups separate on NDVI in the temperate regions and do not separate in the tropics.** In the 11,570-instance sample, the mean NDVI curve per aggregated group in Europe has an amplitude of 0.216 (cereals) to 0.473 (sugar), with peaks spread across timesteps 3 to 5, so the curves are distinguishable by both height and timing. In Sub-Saharan Africa the same amplitudes run from 0.056 (cereals) to 0.204 (beverage and spice), and the group means overlap for the whole window. Median per-instance NDVI amplitude by region tells the same story: 0.572 in North America and 0.489 in Europe against 0.279 in Sub-Saharan Africa and 0.303 in the Caribbean. The ERA5 channel is consistent with the mechanism: the median annual temperature range is 27.2 K in North America and 15.9 K in Europe but only 6.4 K in Sub-Saharan Africa and 2.5 K in the Caribbean.

The fourth panel of the diagnostics figure shows the label-to-pixel offset distribution behind Finding 2b: a tight mode just under one pixel, and a long right tail of instances sampled kilometres from their label.

**Implication.** This is precisely the "harder conditions" the secondary dataset is meant to supply, and it is also a warning. The weak tropical separability is partly real agronomy (year-round growing seasons, multiple cropping) and partly an artefact of the product: a single 10 m pixel, composited to 30-day steps, in the region with the heaviest cloud cover. Any result showing that foundation-model embeddings underperform in Sub-Saharan Africa will have to disentangle those two causes, and the dataset gives no cloud-quality metadata with which to do so. The practical consequence for Phase 2 is that explanations claiming a model attends to "the growing-season peak" will be much harder to validate here than on EuroCropsML, because in the tropical subset there is barely a peak to attend to.

## 7. Comparability with EuroCropsML

This is the section that determines whether CropHarvest is usable rather than merely parallel.

In [18]:
ec = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"]
shared = F["shared_s2_bands"]
cmp = pd.DataFrame([
    ["per-sample unit", "parcel (spatial median over all parcel pixels)", "one 10 m pixel at the label centroid"],
    ["array shape", "(T, 13), T variable", "(12, 18), fixed"],
    ["T, median", "45 (Baltics), 49 (Portugal)", "12, always"],
    ["temporal sampling", "irregular cloud-filtered S2 acquisition dates", "regular 30-day composites"],
    ["window", "calendar year 2021", "360 days ending 1 February, label-specific year"],
    ["sensors", "Sentinel-2 L1C only", "S2 L1C + S1 GRD + ERA5 + SRTM + NDVI"],
    ["S2 bands", f"{len(ec)}", "11 (B1 and B10 dropped)"],
    ["shared S2 bands", f"{len(shared)}", f"{len(shared)}"],
    ["missing data", "variable T, cloud filter applied, no imputation", "imputed with band temporal mean, no mask"],
    ["labels", "176 HCAT classes, one hierarchy", "272 crop types from 27 reconciled schemes"],
    ["geographic units", "3 countries, NUTS-3 regions", "182 countries, 10 regions"],
    ["geolocation integrity", "parcel polygons, aggregated in place", "4.64 % of arrays sampled at the wrong pixel"],
], columns=["property", "EuroCropsML", "CropHarvest"]).set_index("property")
display(cmp)
print("shared Sentinel-2 bands:", ", ".join(shared))
print("EuroCropsML only       :", ", ".join(F["eurocropsml_only_bands"]))
print("CropHarvest only       :", ", ".join(F["cropharvest_only_channels"]))

,EuroCropsML,CropHarvest
property,,
per-sample unit,parcel (spatial median over all parcel pixels),one 10 m pixel at the label centroid
array shape,"(T, 13), T variable","(12, 18), fixed"
"T, median","45 (Baltics), 49 (Portugal)","12, always"
temporal sampling,irregular cloud-filtered S2 acquisition dates,regular 30-day composites
window,calendar year 2021,"360 days ending 1 February, label-specific year"
sensors,Sentinel-2 L1C only,S2 L1C + S1 GRD + ERA5 + SRTM + NDVI
S2 bands,13,11 (B1 and B10 dropped)
shared S2 bands,11,11
missing data,"variable T, cloud filter applied, no imputation","imputed with band temporal mean, no mask"


shared Sentinel-2 bands: B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12
EuroCropsML only       : B01, B10
CropHarvest only       : VV, VH, temperature_2m, total_precipitation, elevation, slope, NDVI


![Comparability](../results/eda/cropharvest/figures/14_comparability_eurocrops.png)

**Finding 15. A shared pipeline is possible, and CropHarvest is the binding constraint on both axes.**

On **input shape**, the two datasets intersect on eleven Sentinel-2 bands: B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12. EuroCropsML additionally has B1 and B10, which CropHarvest discards; CropHarvest additionally has VV, VH, temperature, precipitation, elevation, slope and NDVI, which EuroCropsML does not. To consume both, a shared pipeline must:

1. resample the EuroCropsML irregular series onto a fixed grid, because CropHarvest offers no way to recover sub-monthly detail. The proposal already specifies a monthly-grid raw-feature baseline, so this step exists; it simply becomes mandatory rather than optional, and the grid must be the CropHarvest one, twelve 30-day steps;
2. drop B1 and B10 from the EuroCropsML input, or zero-fill them for CropHarvest;
3. either drop the CropHarvest radar, climate and terrain channels, or provide a per-dataset channel adapter in front of the frozen encoder. The former is the honest choice for a like-for-like comparison and discards seven of eighteen channels;
4. accept that the EuroCropsML parcel median and the CropHarvest centroid pixel are different estimators of the same quantity, with materially different noise, and say so wherever the two are compared.

The binding constraint is **CropHarvest in every case**: twelve timesteps against roughly forty-five, eleven bands against thirteen, and one pixel against a parcel median. Nothing about EuroCropsML limits what can be done with CropHarvest; the reverse is true throughout.

On **temporal alignment**, EuroCropsML is calendar-aligned to 2021. CropHarvest is anchored on 1 February and spans label years 2015 to 2022. The two therefore need per-dataset date handling, and neither aligns with a calendar-year annual embedding product, though EuroCropsML at least aligns with exactly one such product year.

**Finding 16. The TerraTorch backbones can be fed from CropHarvest, but at reduced fidelity.** TerraMind and THOR consume Sentinel-2 imagery; the eleven shared bands cover everything those backbones expect except B1 and B10, which are atmospheric bands rarely used in agricultural tasks. The real loss is temporal: twelve steps is a coarse input for an encoder designed around dense time series, and there is no mask to tell the encoder which steps were interpolated. For Phase 2 this matters twice over, because per-date Integrated Gradients attributions computed over twelve composites answer a much coarser question than the same attributions over forty-five acquisitions.

## 8. Can AlphaEarth and TESSERA even be sampled at these labels?

This is a go or no-go question for the thesis, because two of the four selected foundation models are distributed only as precomputed embeddings. The check is documentary rather than live: Earth Engine is not authenticated in this environment, so coverage is taken from the authoritative catalogue entries and compared against the label years computed above.

**AlphaEarth Foundations**, `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`. The Earth Engine catalogue gives the availability range verbatim as `2017-01-01T00:00:00Z–2025-01-01T00:00:00Z`, one image per calendar year (the 2021 image starts 1 January 2021 and ends 1 January 2022), 10 m resolution, 64 bands in the range -1 to 1, covering global terrestrial land surfaces and shallow water with reduced polar coverage, licensed CC-BY-4.0. Usable years are therefore **2017 to 2024**.

**TESSERA**. The project repository states that embeddings exist globally for **2024** only, that "for regions such as the United States and Europe, embeddings are available from 2017 to 2025", and that coverage is being extended backwards year by year, with an embedding-request form for regions not yet processed. Dimensionality is 128 at 10 m, released CC0.

In [19]:
ALPHAEARTH_YEARS = RUNCFG["alphaearth_years"]
TESSERA_GLOBAL_YEARS = RUNCFG["tessera_global_years"]
yrs = pd.Series(F["dominant_year_counts"]).astype(int)
yrs.index = yrs.index.astype(int)

print("full dataset, by observation-window year:")
display(pd.DataFrame({
    "labels": yrs,
    "AlphaEarth 2017-2024": [y in ALPHAEARTH_YEARS for y in yrs.index],
    "TESSERA global (2024 only)": [y in TESSERA_GLOBAL_YEARS for y in yrs.index],
}))
print(f"AlphaEarth: {F['n_alphaearth_ok']:,} labels in range, "
      f"{F['n_alphaearth_out_of_range']:,} out of range "
      f"({F['n_alphaearth_out_of_range']/len(cat)*100:.1f} %)")
print("  out-of-range years:", F["alphaearth_out_of_range_years"])
print(f"TESSERA: {F['n_tessera_ok']:,} labels within Europe or North America for 2017-2025; "
      f"{F['n_tessera_out_of_coverage']:,} outside current coverage")
print(f"\nCropHarvest label years run {min(yrs.index)}-{max(yrs.index)}; "
      f"TESSERA's only global year is {TESSERA_GLOBAL_YEARS[0]}: overlap = "
      f"{len(set(yrs.index) & set(TESSERA_GLOBAL_YEARS))} years")

n = F["polygon_croptype_n"]
print(f"\nusable polygon crop-type subset (n={n:,}):")
print(f"  AlphaEarth in range : {F['polygon_croptype_alphaearth_ok']:,} "
      f"({F['polygon_croptype_alphaearth_ok']/n*100:.1f} %)")
print(f"  TESSERA in coverage : {F['polygon_croptype_tessera_ok']:,} "
      f"({F['polygon_croptype_tessera_ok']/n*100:.1f} %)")
ssa = cat[(cat.region == "Sub-Saharan Africa") & cat.is_polygon
          & cat.array_usable & cat.is_crop_type]
print(f"\nSub-Saharan Africa, the region the scheme designates for multi-class crop typing "
      f"(n={len(ssa):,}):")
print(f"  AlphaEarth in range : {int(ssa.alphaearth_ok.sum()):,} "
      f"({ssa.alphaearth_ok.mean()*100:.1f} %)")
print(f"  TESSERA in coverage : {int(ssa.tessera_ok.sum()):,} "
      f"({ssa.tessera_ok.mean()*100:.1f} %)")

full dataset, by observation-window year:

,labels,AlphaEarth 2017-2024,TESSERA global (2024 only)
2015,498,False,False
2016,55127,False,False
2017,12945,True,False
2018,14271,True,False
2019,20225,True,False
2020,6264,True,False
2021,4563,True,False


AlphaEarth: 58,268 labels in range, 55,625 out of range (48.8 %)
  out-of-range years: {'2015': 498, '2016': 55127}
TESSERA: 17,856 labels within Europe or North America for 2017-2025; 96,037 outside current coverage

CropHarvest label years run 2015-2021; TESSERA's only global year is 2024: overlap = 0 years

usable polygon crop-type subset (n=20,471):
  AlphaEarth in range : 18,044 (88.1 %)
  TESSERA in coverage : 5,537 (27.0 %)



Sub-Saharan Africa, the region the scheme designates for multi-class crop typing (n=8,213):
  AlphaEarth in range : 8,213 (100.0 %)
  TESSERA in coverage : 0 (0.0 %)


**Finding 17. AlphaEarth: go, but only after a year filter, and with a one-month temporal offset that cannot be removed.**

Across the full dataset, 55,625 labels (48.8 %) have observation windows falling outside AlphaEarth's 2017 to 2024 range: 55,127 in 2016 and 498 in 2015. Almost all of the 2016 block is `geowiki-landcover-2017`, which is point-only and binary-only and is excluded anyway. On the subset the thesis would actually use (polygon, crop-typed, array present, offset check passed), 18,044 of 20,471 labels (88.1 %) fall in range and 2,427 (11.9 %) must be dropped for being pre-2017.

The residual problem is not availability but alignment. A CropHarvest label with `export_end_date` 2019-02-01 describes 1 February 2018 to 27 January 2019. The AlphaEarth 2018 image covers 1 January 2018 to 1 January 2019, so it overlaps that label's window by eleven of twelve months but misses the January tail and includes the January head of a different year. That eleven-twelfths overlap is acceptable, and it is arguably a *better* match than the EuroCropsML case, where an annual embedding averages a season whose informative shoulders sit in a different part of the year. It must nonetheless be stated in every table where AlphaEarth numbers appear, exactly as `CLAUDE.md` already requires.

**Finding 18. TESSERA: no-go in Sub-Saharan Africa on current coverage.**

CropHarvest label years run from 2015 to 2021. TESSERA's only globally available year is 2024. The intersection of those two sets is **empty**. TESSERA embeddings can therefore be sampled at CropHarvest label locations only where the extended regional coverage applies, which the project documents as Europe and the United States for 2017 to 2025. Under that rule 17,856 of 113,893 labels (15.7 %) are covered, and on the usable polygon crop-typed subset 5,537 of 20,471 (27.0 %).

The consequence is severe and specific: **TESSERA covers 0 of the 8,213 usable Sub-Saharan African polygon crop-type labels**, the region Finding 12 identified as the only one in CropHarvest capable of supporting a multi-class crop-typing task. AlphaEarth, by contrast, covers all 8,213 of them, because every Sub-Saharan African label in that subset has an observation window from 2017 onwards. The entire reason for including CropHarvest is to reach smallholder tropical systems, and one of the four selected foundation models cannot be evaluated there at all without submitting a regional embedding request and waiting for it to be processed. Two of the four models would be evaluated on the secondary dataset's headline region; two would not, which breaks the like-for-like comparison that RQ1 depends on.

This is the single strongest technical argument against retaining CropHarvest in its current form, and it is not a fixable property of the dataset: it is a property of TESSERA's release schedule intersected with CropHarvest's label years. Requesting Sub-Saharan African embeddings for 2018 and 2019 from the TESSERA project is the one action that would resolve it, and it should be attempted early because it has a lead time outside the student's control.

## 9. Verdict on fitness for purpose

**CropHarvest should be retained, but with its role narrowed and stated explicitly, and a fallback identified.**

The case for retaining it:

1. It is the only readily available dataset that places the pipeline in smallholder tropical agriculture at scale, with 18,025 polygon labels in Sub-Saharan Africa at a median field size of 0.72 ha.
2. Sub-Saharan Africa supports a real twelve-class crop-typing task over 8,213 usable polygon labels, which is close to the fifteen to twenty classes per country the proposal specifies for EuroCropsML, and every one of those labels is inside AlphaEarth's coverage.
3. Acquisition is trivial: 830 MB, four minutes of download and extraction, a permissive CC-BY-SA-4.0 licence, and no dependency on the conflicting Python package.
4. The failure modes it exposes are themselves findings. Weak tropical NDVI separability, a fixed February anchor, and single-pixel sampling are exactly the conditions under which the transparency and uncertainty work in Phase 2 becomes interesting rather than decorative.

The case against, and the honest statement of what it cannot do:

1. **It is not a per-parcel dataset.** The released feature array is one 10 m pixel at the polygon centroid. The polygon restriction gives label-footprint metadata, not parcel aggregation. A direct like-for-like comparison against EuroCropsML's parcel medians is therefore not possible from the released artefacts.
2. **The cross-region transfer experiment does not survive.** No two regions share more than two crop classes at a usable sample size. The EuroCropsML transnational protocol has no faithful analogue here at crop level, only at aggregated-group level and only with Sub-Saharan Africa on one side.
3. **TESSERA cannot be sampled where it matters most**, as Finding 18 establishes.
4. **4.64 % of arrays are sampled at the wrong place**, silently, and the filter for this is not documented anywhere upstream. Two whole smallholder sources are lost to it.
5. **Twelve monthly composites is a coarse input** for encoders designed around dense time series, and there is no validity mask to tell them which steps are imputed.

**Recommended framing.** Keep CropHarvest, and use it for a narrower purpose than the proposal currently implies: a **single-region, smallholder, tropical stress test of the label-efficiency result**, run in Sub-Saharan Africa on the twelve-class scheme, with cross-region transfer reported only at the aggregated-group level and flagged as a two-to-eight-class problem rather than a peer of the EuroCropsML protocol. Do not present it as a second full benchmark.

**Alternatives, if that narrowing is unacceptable.** The candidates worth checking, compared on the five properties that matter here:

| Dataset | Smallholder | Tropical | Polygon labels | S2 time series | Licence |
|---|---|---|---|---|---|
| **Lacuna Fund Global Crop Type** | strong, its explicit purpose | yes, Africa and Asia | yes, field boundaries | supplied or derivable | open, varies by contribution |
| **Radiant MLHub African crop sets** (Kenya, Tanzania, Rwanda) | strong | yes | yes, field boundaries | yes, per-field S2 series | CC-BY-4.0, hosting has moved from Radiant |
| **WorldCereal 2021 reference** | mixed, global harmonisation | partly | mixed points and polygons | not bundled, must be extracted | CC-BY-4.0 |
| **Sen4AgriNet** | no, Mediterranean commercial | no | yes, parcel level | yes, dense S2 patches | open |
| **PASTIS-R** | no, French commercial | no | yes, parcel level | yes, S2 and S1 | open |
| **ZueriCrop** | no, Swiss commercial | no | yes, parcel level | yes, dense S2 | open |
| **Sen12Crop** | no | no | parcel level | S1 and S2 | open |

The last four are all European, commercial-scale and temperate. They would make excellent *additional primary* datasets but they do nothing that EuroCropsML does not already do, so they are not substitutes for a secondary dataset whose entire purpose is to leave that regime.

The two genuine alternatives are therefore the **Radiant MLHub African smallholder sets** and the **Lacuna Fund Global Crop Type** collection. Both supply field-boundary polygons rather than centroid points, which removes CropHarvest's most serious limitation and restores a true per-parcel aggregation comparable with EuroCropsML. Both would need their own Sentinel-2 extraction, which CropHarvest supplies ready-made, so the cost is one extraction pipeline against the gain of a per-parcel, polygon-aggregated, smallholder tropical dataset. Given that the thesis already needs an Earth Engine extraction pipeline for AlphaEarth, that cost is lower than it appears. It is worth noting that CropHarvest's own `tanzania`, `uganda` and `kenya` sources are derived from those same Radiant MLHub collections, and that two of the three are precisely the ones broken by Finding 2b, so going to the original source would recover data the harmonised version has lost.

**Recommendation.** Proceed with CropHarvest now, on the narrowed Sub-Saharan Africa framing, because it is available today and unblocks Phase 1 immediately. In parallel, submit a TESSERA embedding request for Sub-Saharan Africa for 2018 and 2019, and assess the Lacuna Fund Global Crop Type collection as the replacement if the per-parcel limitation proves to bite in practice.

## Prior foundation-model evaluations on CropHarvest

<!-- PLACEHOLDER: to be filled from docs/research/prior_gfm_use_cropharvest.md -->

## 10. Headline conclusions

1. **The download is small and the format is honest.** 830 MB, 113,893 labels and 87,464 arrays, both MD5-verified against Zenodo record 10251170. The band order, the twelve 30-day timesteps and the 1 February anchor were all confirmed against the files rather than the documentation. The `cropharvest` package was deliberately not installed because it pins `pandas<2.0.0`.
2. **The single most important property is that the feature array is one pixel, not a parcel.** The polygon-labelled subset provides label footprints and area, but the array itself is the 10 m pixel at the centroid. Statements comparing CropHarvest with EuroCropsML must say so every time.
3. **A previously undocumented defect affects 4,062 arrays, 4.64 % of all 87,464.** The sampled pixel lies beyond one pixel diagonal from its label, and in 3.33 % of cases beyond 1 km, because the exporter snaps out-of-extent labels to the edge of a shared GeoTIFF rather than rejecting them. It is confined to three sources, of which `tanzania` (379 of 379) and `uganda` (225 of 225) are entirely affected. It is detectable from the `label_lat` and `instance_lat` attributes and must be filtered before any use.
4. **Only 31.0 % of labels name a crop.** 53,206 labels come from eleven binary-only sources, 26,429 have no feature array, and a further 14,552 name land cover rather than a crop. The FAO harmonisation has documented defects affecting 4,784 labels, including cotton pooled into `other` and French labels split by land tenure rather than by crop.
5. **The polygon restriction is cheap and beneficial**, retaining 72.0 % of the crop-typed pool and removing both sources flagged for geolocation precision. After every filter, 20,471 usable polygon crop-type labels remain.
6. **The cross-region transfer experiment does not transfer.** No region pair shares more than two crop classes at 150 labels each, against eleven shared classes across three countries in EuroCropsML. Aggregation to agronomic groups is the only route to a multi-class cross-region experiment.
7. **Sub-Saharan Africa is the whole value proposition.** It is the only region supporting a multi-class crop-typing task (twelve classes, 15,003 crop-typed labels, 18,025 polygons), it has the smallest fields, and it is the region where NDVI separability between crop groups is weakest, with mean-curve amplitudes of 0.056 to 0.204 against 0.216 to 0.473 in Europe.
8. **AlphaEarth is a go after a pre-2017 filter**, retaining 88.1 % of the usable polygon crop-typed subset, with an eleven-twelfths window overlap that must be declared.
9. **TESSERA is a no-go in Sub-Saharan Africa on current coverage**, covering 0 of the 8,213 usable labels there, because CropHarvest's label years end in 2021 and TESSERA's only global year is 2024. This affects precisely the region that justifies the dataset's inclusion, and it would leave two of the four selected models unevaluated on the secondary dataset's headline experiment.
10. **Spatial blocking must be at least 100 m and preferably 1 km.** 9.2 % of labels have a neighbour within 10 m and 53.8 % within 1 km, almost entirely within rather than across source datasets, so the hazard is autocorrelation, not duplicate records.

### Direct next steps

- Implement the label-to-pixel offset filter from Finding 2b as the first step of the CropHarvest loader, before anything else touches the arrays. `results/eda/cropharvest/offset_scan.parquet` already holds the per-instance flag.
- Submit a TESSERA embedding request for Sub-Saharan Africa covering 2018 and 2019, which are the two densest label years in the usable polygon crop-typed subset. This has an external lead time and should be the first action taken.
- Settle the open K definition in `CLAUDE.md` before building splits. The 150-label floor adopted here presumes K is expressed in samples per class with a maximum of 100; if K becomes a percentage grid, the floor and therefore the surviving class list both change.
- Build the Sub-Saharan Africa twelve-class split from `configs/class_scheme_cropharvest.yaml` with 1 km spatial blocks, and confirm that every class still clears 150 labels after blocking.
- Extract AlphaEarth embeddings at the 18,044 in-range usable polygon centroids, matching each label to the AlphaEarth year equal to its `dominant_year`, and record the eleven-twelfths offset in the output metadata.
- Assess the Lacuna Fund Global Crop Type collection against the same questions this notebook asked, since it is the strongest replacement candidate and the one that would restore per-parcel aggregation.